# Compute mean change confidence intervals

Full temporal coverage: compute change confidence intervals for every index configured in `configs/change_ci.yaml`, using all same-month consecutive-year pairs from 2019 to 2025 across all study areas. Results are saved under `outputs/change_ci/<INDEX>/`.

In [1]:
from pathlib import Path

import pandas as pd
import rasterio

from change_ci import AnalysisConfig, ChangeCIAnalysis, build_pairs, discover_rasters

ROOT = Path.cwd().parent
assert (ROOT / "pyproject.toml").exists(), "Run notebooks from the notebooks/ folder"

## Analysis configuration

In [2]:
config = AnalysisConfig.from_yaml(
    ROOT / "configs" / "change_ci.yaml", project_root=ROOT
)
config.to_dict()

{'data_root': '/home/tshihadah/code/wadi-gaza-research/outputs/composites',
 'output_dir': '/home/tshihadah/code/wadi-gaza-research/outputs/change_ci',
 'study_areas': None,
 'months': ('01', '05', '09', '11'),
 'year_range': (2019, 2025),
 'index_bands': ('NDVI', 'GNDVI', 'SAVI', 'NDRE'),
 'engine': 'fft',
 'tau': 0.05,
 'tau_sensitivity': (0.02, 0.05, 0.1),
 'alpha': 0.05,
 'max_lag_fraction': 0.25,
 'min_n_eff': 30}

## Discover all same-month consecutive-year pairs

Expected coverage: 168 date pairs per index (seven study areas × four months × six year transitions).

In [3]:
raster_records = discover_rasters(config.data_root)
missing_bands = []
for record in raster_records:
    with rasterio.open(record.path) as raster:
        missing = set(config.index_bands) - set(raster.descriptions)
    if missing:
        missing_bands.append((record.path, sorted(missing)))
if missing_bands:
    raise ValueError(f'Configured index bands missing from composites: {missing_bands[:10]}')
pair_specs = build_pairs(raster_records, config)
study_areas = config.study_areas or tuple(sorted({record.study_area for record in raster_records}))
expected_pairs = len(study_areas) * len(config.months) * (config.year_range[1] - config.year_range[0])
if len(pair_specs) != expected_pairs:
    raise ValueError(f'Expected {expected_pairs} date pairs, discovered {len(pair_specs)}')
pair_table = pd.DataFrame([
    {
        'study_area': pair.study_area,
        'month': pair.month,
        'year_start': pair.year_start,
        'year_end': pair.year_end,
        'earlier_path': pair.earlier_path,
        'later_path': pair.later_path,
    }
    for pair in pair_specs
])
pair_table

,study_area,month,year_start,year_end,earlier_path,later_path
0,AlAhrashProtectorate,01,2019,2020,/home/tshihadah/code/wadi-gaza-research/output...,/home/tshihadah/code/wadi-gaza-research/output...
1,AlAhrashProtectorate,01,2020,2021,/home/tshihadah/code/wadi-gaza-research/output...,/home/tshihadah/code/wadi-gaza-research/output...
2,AlAhrashProtectorate,01,2021,2022,/home/tshihadah/code/wadi-gaza-research/output...,/home/tshihadah/code/wadi-gaza-research/output...
3,AlAhrashProtectorate,01,2022,2023,/home/tshihadah/code/wadi-gaza-research/output...,/home/tshihadah/code/wadi-gaza-research/output...
4,AlAhrashProtectorate,01,2023,2024,/home/tshihadah/code/wadi-gaza-research/output...,/home/tshihadah/code/wadi-gaza-research/output...
...,...,...,...,...,...,...
163,WadiGaza,11,2020,2021,/home/tshihadah/code/wadi-gaza-research/output...,/home/tshihadah/code/wadi-gaza-research/output...
164,WadiGaza,11,2021,2022,/home/tshihadah/code/wadi-gaza-research/output...,/home/tshihadah/code/wadi-gaza-research/output...
165,WadiGaza,11,2022,2023,/home/tshihadah/code/wadi-gaza-research/output...,/home/tshihadah/code/wadi-gaza-research/output...
166,WadiGaza,11,2023,2024,/home/tshihadah/code/wadi-gaza-research/output...,/home/tshihadah/code/wadi-gaza-research/output...


## Run and save

In [4]:
analysis = ChangeCIAnalysis(config)
results, correlograms = analysis.run(pair_specs)
analysis.save(results, correlograms)
print(f'Saved {len(results)} result rows and {len(correlograms)} correlogram rows under {config.output_dir}')
index_summary = results.groupby('index').agg(
    result_rows=('index', 'size'),
    failed_pairs=('error', lambda values: values.fillna('').ne('').sum()),
)
display(index_summary)
expected_rows = len(pair_specs) * len(config.tau_sensitivity)
index_rows = index_summary['result_rows'].reindex(config.index_bands, fill_value=0)
if not index_rows.eq(expected_rows).all() or index_summary['failed_pairs'].sum():
    raise RuntimeError('At least one index has incomplete or failed pair results; inspect saved result tables.')

Raster pixels are not square: (9.999368875690768, 9.989040393093362)


Raster pixels are not square: (9.999368875690768, 9.989040393093362)


Raster pixels are not square: (9.999368875690768, 9.989040393093362)


Raster pixels are not square: (9.999368875690768, 9.989040393093362)


Raster pixels are not square: (9.999368875690768, 9.989040393093362)


Raster pixels are not square: (9.999368875690768, 9.989040393093362)


... (664 outputs left out)

Raster pixels are not square: (10.000130544932942, 10.00695479149712)


Raster pixels are not square: (10.000130544932942, 10.00695479149712)


Saved 2016 result rows and 50304 correlogram rows under /home/tshihadah/code/wadi-gaza-research/outputs/change_ci


,result_rows,failed_pairs
index,,
GNDVI,504,0
NDRE,504,0
NDVI,504,0
SAVI,504,0


## Results and review flags

In [5]:
primary = results.loc[results['is_primary_tau']].copy()
summary_columns = [
    'study_area', 'index', 'month', 'year_start', 'year_end', 'n', 'mean_change', 'sd',
    'range_px', 'range_map_units', 'n_eff', 'ci_low', 'ci_high',
    'naive_ci_low', 'naive_ci_high', 'flag_no_range', 'flag_low_n_eff', 'error',
]
display(primary[summary_columns].sort_values(['index', 'study_area', 'month', 'year_start']))

review = primary.loc[
    primary['flag_no_range']
    | primary['flag_low_n_eff']
    | primary['error'].fillna('').ne('')
]
print(f'{len(review)} primary-threshold result(s) flagged or failed.')
display(review[summary_columns].sort_values(['index', 'study_area', 'month', 'year_start']))

,study_area,index,month,year_start,year_end,n,mean_change,sd,range_px,range_map_units,n_eff,ci_low,ci_high,naive_ci_low,naive_ci_high,flag_no_range,flag_low_n_eff,error
505,AlAhrashProtectorate,GNDVI,01,2019,2020,33848,0.024480,0.033388,22.0,219.986115,152.050907,0.019130,0.029830,0.024124,0.024836,False,False,
508,AlAhrashProtectorate,GNDVI,01,2020,2021,33848,-0.023397,0.046879,47.0,469.970337,56.365504,-0.035909,-0.010885,-0.023896,-0.022897,False,False,
511,AlAhrashProtectorate,GNDVI,01,2021,2022,33848,0.030623,0.036258,43.0,429.972862,59.650284,0.021229,0.040018,0.030237,0.031010,False,False,
514,AlAhrashProtectorate,GNDVI,01,2022,2023,33848,-0.011622,0.033466,28.0,279.982329,88.665797,-0.018685,-0.004558,-0.011978,-0.011265,False,False,
517,AlAhrashProtectorate,GNDVI,01,2023,2024,33848,-0.002944,0.037592,36.0,359.977280,76.960935,-0.011479,0.005590,-0.003345,-0.002544,False,False,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1498,WadiGaza,SAVI,11,2020,2021,28413,-0.000587,0.060779,12.0,120.001567,419.551728,-0.006420,0.005246,-0.001294,0.000120,False,False,
1501,WadiGaza,SAVI,11,2021,2022,28413,-0.002835,0.059748,7.0,70.000914,854.788034,-0.006846,0.001176,-0.003530,-0.002140,False,False,
1504,WadiGaza,SAVI,11,2022,2023,28413,0.000341,0.063785,8.0,80.001044,740.449899,-0.004261,0.004943,-0.000401,0.001082,False,False,
1507,WadiGaza,SAVI,11,2023,2024,28413,-0.076759,0.095818,26.0,260.003394,101.268649,-0.095649,-0.057870,-0.077874,-0.075645,False,False,


92 primary-threshold result(s) flagged or failed.


,study_area,index,month,year_start,year_end,n,mean_change,sd,range_px,range_map_units,n_eff,ci_low,ci_high,naive_ci_low,naive_ci_high,flag_no_range,flag_low_n_eff,error
529,AlAhrashProtectorate,GNDVI,05,2021,2022,33848,0.013815,0.030990,NaN,NaN,NaN,NaN,NaN,0.013485,0.014145,True,False,
535,AlAhrashProtectorate,GNDVI,05,2023,2024,33848,-0.025911,0.040828,NaN,NaN,NaN,NaN,NaN,-0.026346,-0.025476,True,False,
538,AlAhrashProtectorate,GNDVI,05,2024,2025,33848,0.006873,0.022264,NaN,NaN,NaN,NaN,NaN,0.006636,0.007110,True,False,
553,AlAhrashProtectorate,GNDVI,09,2023,2024,33848,-0.015606,0.032686,NaN,NaN,NaN,NaN,NaN,-0.015954,-0.015258,True,False,
556,AlAhrashProtectorate,GNDVI,09,2024,2025,33848,0.007149,0.020294,NaN,NaN,NaN,NaN,NaN,0.006933,0.007365,True,False,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1402,NahalRubin,SAVI,05,2024,2025,7811,-0.037225,0.038402,24.0,240.409076,29.750175,-0.05163,-0.02282,-0.038077,-0.036373,False,True,
1411,NahalRubin,SAVI,09,2021,2022,7714,0.009504,0.032054,NaN,NaN,NaN,NaN,NaN,0.008789,0.010220,True,False,
1423,NahalRubin,SAVI,11,2019,2020,7860,0.007999,0.041365,NaN,NaN,NaN,NaN,NaN,0.007084,0.008914,True,False,
1426,NahalRubin,SAVI,11,2020,2021,7860,-0.033058,0.046373,NaN,NaN,NaN,NaN,NaN,-0.034084,-0.032033,True,False,
